
# 3I/ATLAS Research Notebook
This notebook pulls **JPL Horizons ephemerides** and **MAST** search results for **3I/ATLAS (C/2025 N1 (ATLAS))**, builds a tidy dataset catalog (CSV/SQLite), and creates quick‑look plots for **trajectory** and **brightness timeline**.


In [5]:

# Parameters
OBJECT_NAMES = ["C/2025 N1"]  # Correct JPL Horizons identifier (without (ATLAS))

# Rolling 7-day window (relative dates)
today = datetime.now()
DATE_END = today.strftime("%Y-%m-%d")
DATE_START = (today - timedelta(days=7)).strftime("%Y-%m-%d")

STEP = "1d"  # ephemerides cadence

# Output paths
OUT_DIR = "outputs_3I_ATLAS"
os.makedirs(OUT_DIR, exist_ok=True)

CSV_EPHEMERIDES = os.path.join(OUT_DIR, "ephemerides.csv")
CSV_MAST        = os.path.join(OUT_DIR, "mast_catalog.csv")
CSV_DATASETS    = os.path.join(OUT_DIR, "dataset_catalog.csv")
SQLITE_DB       = os.path.join(OUT_DIR, "catalog.sqlite")
FIG_TRAJECTORY  = os.path.join(OUT_DIR, "trajectory_ra_dec.png")
FIG_BRIGHTNESS  = os.path.join(OUT_DIR, "brightness_timeline.png")


In [1]:

# Imports (with optional astroquery)
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Try astroquery for Horizons + MAST
HAVE_ASTROQUERY = True
try:
    from astroquery.jplhorizons import Horizons
    from astroquery.mast import Observations
except Exception as e:
    HAVE_ASTROQUERY = False
    print("astroquery not available; will use offline fallbacks.")


## Pull JPL Horizons Ephemerides

In [6]:

def fetch_ephemerides(object_names, date_start, date_end, step):
    if not HAVE_ASTROQUERY:
        raise RuntimeError("astroquery is not available. Cannot fetch ephemerides.")
    
    errors = []
    for name in object_names:
        try:
            obj = Horizons(id=name, location='500@10',  # heliocentric frame proxy: use Sun barycenter? Using Earth L1 (500@10) is fine for apparent RA/Dec
                           epochs={'start': date_start, 'stop': date_end, 'step': step})
            eph = obj.ephemerides()
            if len(eph) > 0:
                df = eph.to_pandas()
                df['object_name'] = name
                print(f"Successfully retrieved ephemerides for {name}")
                return df
        except Exception as e:
            error_msg = f"Failed to query {name}: {e}"
            print(error_msg)
            errors.append(error_msg)
            continue
    
    # If we get here, all queries failed
    raise RuntimeError(f"Could not retrieve ephemerides for any object. Errors:\n" + "\n".join(errors))

ephem_df = fetch_ephemerides(OBJECT_NAMES, DATE_START, DATE_END, STEP)
ephem_df.to_csv(CSV_EPHEMERIDES, index=False)
ephem_df.head()


Successfully retrieved ephemerides for C/2025 N1


,targetname,datetime_str,datetime_jd,M1,solar_presence,k1,interfering_body,RA,DEC,RA_app,...,r_rate_3sigma,SBand_3sigma,XBand_3sigma,DoppDelay_3sigma,true_anom,hour_angle,alpha_true,PABLon,PABLat,object_name
0,ATLAS (C/2025 N1),2025-Nov-02 00:00,2460981.5,12.3,,4.5,,188.92806,-0.01791,188.09054,...,0.026423,371.51,1350.23,0.354392,5.8573,<NA>,0.0130,188.2154,3.5232,C/2025 N1
1,ATLAS (C/2025 N1),2025-Nov-03 00:00,2460982.5,12.3,,4.5,,187.37789,0.53998,186.44975,...,0.027223,378.17,1374.44,0.364035,7.5056,<NA>,0.0130,186.5678,3.4244,C/2025 N1
2,ATLAS (C/2025 N1),2025-Nov-04 00:00,2460983.5,12.3,,4.5,,185.83747,1.09387,184.81899,...,0.028009,384.35,1396.92,0.374030,9.1432,<NA>,0.0129,184.9312,3.3234,C/2025 N1
3,ATLAS (C/2025 N1),2025-Nov-05 00:00,2460984.5,12.3,,4.5,,184.30855,1.64263,183.20041,...,0.028781,390.06,1417.67,0.384363,10.7680,<NA>,0.0129,183.3078,3.2205,C/2025 N1
4,ATLAS (C/2025 N1),2025-Nov-06 00:00,2460985.5,12.3,,4.5,,182.79280,2.18522,181.59606,...,0.029537,395.30,1436.70,0.395021,12.3779,<NA>,0.0128,181.6996,3.1160,C/2025 N1


## Query MAST for HST/JWST Observations

In [7]:

def query_mast(object_names, date_start, date_end, max_records=100, timeout=60):
    if not HAVE_ASTROQUERY:
        raise RuntimeError("astroquery is not available. Cannot query MAST.")
    
    # Set timeout for astroquery
    from astroquery.mast import Conf
    Conf.timeout = timeout
    
    # Try by target name; also restrict to HST/JWST missions
    all_obs = []
    errors = []
    
    for i, name in enumerate(object_names):
        print(f"Querying MAST for object {i+1}/{len(object_names)}: {name}")
        print(f"  Date range: {date_start} to {date_end}")
        print(f"  Timeout: {timeout}s, Max records: {max_records}")
        
        try:
            # Add timeout and page size limits
            obs = Observations.query_criteria(
                target_name=name,
                obs_collection=["HST", "JWST"],
                dataproduct_type=["image","spectrum"],
                t_min=[date_start, None],
                t_max=[None, date_end],
                pagesize=max_records  # Limit results per query
            )
            
            print(f"  -> Found {len(obs)} observations for {name}")
            
            if len(obs) > 0:
                all_obs.append(obs.to_pandas())
        except Exception as e:
            error_msg = f"MAST query failed for {name}: {e}"
            print(error_msg)
            errors.append(error_msg)
            continue

    if all_obs:
        df = pd.concat(all_obs, ignore_index=True)
        print(f"\nTotal observations across all objects: {len(df)}")
        return df
    else:
        # If we get here, no observations were found
        raise RuntimeError(f"No MAST observations found for any object. Errors:\n" + "\n".join(errors) if errors else "No observations matched the query criteria.")

mast_df = query_mast(OBJECT_NAMES, DATE_START, DATE_END, max_records=50, timeout=120)
mast_df.to_csv(CSV_MAST, index=False)
mast_df.head()


Querying MAST for object 1/1: C/2025 N1
  Date range: 2025-11-02 to 2025-11-09
  Timeout: 120s, Max records: 50
MAST query failed for C/2025 N1: Thread was being aborted.
MAST query failed for C/2025 N1: Thread was being aborted.


RuntimeError: No MAST observations found for any object. Errors:
MAST query failed for C/2025 N1: Thread was being aborted.

## Build Tidy Dataset Catalog (CSV + SQLite)

In [ ]:

# Normalize/prepare
eph_min = ephem_df[['datetime_str', 'RA', 'DEC', 'r', 'delta', 'V', 'object_name']].copy()
eph_min.rename(columns={'datetime_str':'datetime_utc'}, inplace=True)
eph_min['source'] = 'JPL Horizons'

mast_min = mast_df[['obsid','obs_collection','instrument_name','target_name','t_observe_start','t_observe_end','filters','proposal_id','dataURL']].copy()
mast_min.rename(columns={
    't_observe_start':'datetime_obs_start_utc',
    't_observe_end':'datetime_obs_end_utc',
    'obs_collection':'mission'
}, inplace=True)
mast_min['source'] = 'MAST'

# Persist as separate CSVs (already saved ephemerides & mast above)
# Build a combined catalog index for convenience
dataset_catalog = {
    'ephemerides_csv': CSV_EPHEMERIDES,
    'mast_csv': CSV_MAST
}
catalog_df = pd.DataFrame([dataset_catalog])
catalog_df.to_csv(CSV_DATASETS, index=False)

# Save to SQLite
conn = sqlite3.connect(SQLITE_DB)
eph_min.to_sql("ephemerides", conn, if_exists="replace", index=False)
mast_min.to_sql("mast_observations", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

print("Saved:")
print(" -", CSV_EPHEMERIDES)
print(" -", CSV_MAST)
print(" -", CSV_DATASETS)
print(" -", SQLITE_DB)


Saved:
 - outputs_3I_ATLAS\ephemerides.csv
 - outputs_3I_ATLAS\mast_catalog.csv
 - outputs_3I_ATLAS\dataset_catalog.csv
 - outputs_3I_ATLAS\catalog.sqlite


## Quick‑Look Plots

In [8]:

# Ensure datetime conversion
eph_plot = ephem_df.copy()
if 'datetime_str' in eph_plot.columns:
    eph_plot['dt'] = pd.to_datetime(eph_plot['datetime_str'])
elif 'datetime_utc' in eph_plot.columns:
    eph_plot['dt'] = pd.to_datetime(eph_plot['datetime_utc'])
else:
    # Horizons pandas may include 'datetime_str' already; default fallback
    eph_plot['dt'] = pd.to_datetime(eph_plot.iloc[:,0], errors='coerce')

# 1) Trajectory: RA vs Dec
plt.figure()
plt.plot(eph_plot['RA'], eph_plot['DEC'], marker='o')
plt.xlabel("Right Ascension (deg)")
plt.ylabel("Declination (deg)")
plt.title("3I/ATLAS Trajectory (Sky Plane)")
plt.gca().invert_xaxis()  # RA increases to the left convention
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_TRAJECTORY, dpi=150)
plt.close()

# 2) Brightness timeline: V magnitude vs time
plt.figure()
plt.plot(eph_plot['dt'], eph_plot['V'], marker='o')
plt.gca().invert_yaxis()  # lower magnitudes = brighter
plt.xlabel("Date (UTC)")
plt.ylabel("V Magnitude")
plt.title("3I/ATLAS Brightness Timeline (V)")
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_BRIGHTNESS, dpi=150)
plt.close()

FIG_TRAJECTORY, FIG_BRIGHTNESS


('outputs_3I_ATLAS\\trajectory_ra_dec.png',
 'outputs_3I_ATLAS\\brightness_timeline.png')

## Notes
- When you have internet access, re‑run all cells to pull live data.
- To refine MAST queries, adjust `Observations.query_criteria` with specific missions/instruments or RA/Dec cones.